# Major Project: Seasonal Agriculture Performance Analysis

**VOIS AICTE Batch 2026–2027**

### Project focus
This notebook analyzes the provided agricultural dataset to identify seasonal patterns, trends, relationships, resource usage differences, economic outcomes, and unusual observations.

The analysis follows the project brief: **data understanding → cleaning → exploratory analysis → seasonal comparison → relationship analysis → visualization → findings → recommendations**.

## 1. Problem Statement

Agricultural performance is affected by seasonal environmental conditions, farming practices, resource availability, and market conditions. The objective is to analyze the given data and investigate how agricultural performance differs across **Kharif, Rabi, and Zaid** seasons.

The project brief asks us to identify meaningful patterns, trends, relationships, variations, and evidence-based insights for better seasonal agricultural planning.

## 2. Objectives

- Explore and understand the dataset.
- Clean and prepare the data.
- Compare agricultural performance across seasons.
- Analyze environmental and resource-related variables.
- Study relationships between conditions and agricultural outcomes.
- Compare crops, states, districts, and irrigation methods where useful.
- Identify unusual observations and seasonal variations.
- Use appropriate statistical and visualization techniques.
- Develop evidence-based conclusions and recommendations.

In [ ]:
# Cell 1: Import required libraries

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

# Plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries imported successfully.")

In [ ]:
# Cell 2: Load the dataset

file_path = "seasonal_agriculture_performance_dataset.csv"
df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
# Cell 3: Preview the data

display(df.head())
display(df.tail())

In [ ]:
# Cell 4: Understand the structure of the dataset

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nBasic information:")
df.info()

In [ ]:
# Cell 5: Check missing values and duplicates

missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

print("Missing values:")
display(missing.to_frame("Missing Count"))

print("Total missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

## 3. Data Cleaning

The dataset contains missing values in **Rainfall_mm, Soil_Moisture_pct, and Yield_Tonnes_Ha**. For numerical analysis, missing numeric values are filled using the **median**, which is less sensitive to extreme observations than the mean.

Duplicate rows are removed if any are present. The original categorical values are retained.

In [ ]:
# Cell 6: Clean the dataset

df_clean = df.copy()

numeric_cols = df_clean.select_dtypes(include=np.number).columns

for col in numeric_cols:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
after = len(df_clean)

print("Missing values after cleaning:", df_clean.isnull().sum().sum())
print("Duplicate rows removed:", before - after)
print("Cleaned dataset shape:", df_clean.shape)

In [ ]:
# Cell 7: Descriptive statistics

display(df_clean.describe().T.round(2))

In [ ]:
# Cell 8: Explore categorical variables

categorical_cols = ["State", "District", "Crop", "Season", "Irrigation_Method"]

for col in categorical_cols:
    print(f"\n{col}:")
    print(df_clean[col].value_counts())

## 4. Seasonal Distribution

First, we examine how many records belong to each season. This gives context before comparing averages because the number of observations is not identical across seasons.

In [ ]:
# Cell 9: Count of farms by season — Seaborn count plot

plt.figure(figsize=(9, 5))
ax = sns.countplot(data=df_clean, x="Season", order=df_clean["Season"].value_counts().index)
ax.set_title("Number of Farm Records by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Number of Records")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10: Seasonal performance summary

season_summary = (
    df_clean.groupby("Season")
    .agg(
        Farms=("Farm_ID", "count"),
        Avg_Yield=("Yield_Tonnes_Ha", "mean"),
        Avg_Production=("Production_Tonnes", "mean"),
        Avg_Revenue=("Revenue_INR", "mean"),
        Avg_Profit=("Profit_INR", "mean"),
        Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
        Avg_Disease_Risk=("Disease_Pest_Risk_pct", "mean")
    )
    .round(2)
)

display(season_summary)

## 5. Agricultural Performance Across Seasons

The following visualizations compare yield, revenue, and profit across seasons. These are central performance indicators for the project.

In [ ]:
# Cell 11: Average yield by season — Matplotlib bar chart

avg_yield = df_clean.groupby("Season")["Yield_Tonnes_Ha"].mean().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
plt.bar(avg_yield.index, avg_yield.values)
plt.title("Average Yield by Season")
plt.xlabel("Season")
plt.ylabel("Average Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: Average profit by season — Seaborn bar plot

avg_profit = df_clean.groupby("Season")["Profit_INR"].mean().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=avg_profit.index, y=avg_profit.values)
plt.title("Average Profit by Season")
plt.xlabel("Season")
plt.ylabel("Average Profit (INR)")
plt.ticklabel_format(style="plain", axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13: Revenue and profit comparison

economic = df_clean.groupby("Season")[["Revenue_INR", "Profit_INR"]].mean().round(2)

ax = economic.plot(kind="bar", figsize=(10, 6))
ax.set_title("Average Revenue and Profit by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Amount (INR)")
ax.ticklabel_format(style="plain", axis="y")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(economic)

In [ ]:
# Cell 14: Yield distribution by season — Seaborn box plot

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_clean, x="Season", y="Yield_Tonnes_Ha")
plt.title("Yield Distribution Across Seasons")
plt.xlabel("Season")
plt.ylabel("Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()

### Interpretation point

The box plot is useful because averages alone can hide variability. It shows the median, spread, and potential extreme yield observations for each season.

In [ ]:
# Cell 15: Crop-wise average yield

crop_yield = (
    df_clean.groupby("Crop")["Yield_Tonnes_Ha"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(11, 6))
sns.barplot(x=crop_yield.values, y=crop_yield.index)
plt.title("Average Yield by Crop")
plt.xlabel("Average Yield (Tonnes/Ha)")
plt.ylabel("Crop")
plt.tight_layout()
plt.show()

display(crop_yield.round(2).to_frame("Average Yield (Tonnes/Ha)"))

In [ ]:
# Cell 16: Crop-season yield comparison — heatmap

crop_season = pd.pivot_table(
    df_clean,
    values="Yield_Tonnes_Ha",
    index="Crop",
    columns="Season",
    aggfunc="mean"
)

plt.figure(figsize=(10, 7))
sns.heatmap(crop_season, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Average Yield by Crop and Season")
plt.xlabel("Season")
plt.ylabel("Crop")
plt.tight_layout()
plt.show()

## 6. Environmental Conditions

Seasonal agricultural outcomes can be associated with rainfall, temperature, humidity, sunlight, soil moisture, and soil pH. We examine these variables before studying their relationships with yield.

In [ ]:
# Cell 17: Seasonal environmental averages

environment = df_clean.groupby("Season")[
    ["Rainfall_mm", "Avg_Temperature_C", "Humidity_pct",
     "Sunlight_Hours_Day", "Soil_Moisture_pct", "Soil_pH"]
].mean().round(2)

display(environment)

In [ ]:
# Cell 18: Seasonal rainfall — Matplotlib line chart

rainfall = df_clean.groupby("Season")["Rainfall_mm"].mean()

plt.figure(figsize=(9, 5))
plt.plot(rainfall.index, rainfall.values, marker="o", linewidth=2)
plt.title("Average Rainfall Across Seasons")
plt.xlabel("Season")
plt.ylabel("Rainfall (mm)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 19: Rainfall distribution — Seaborn histogram

plt.figure(figsize=(10, 6))
sns.histplot(data=df_clean, x="Rainfall_mm", hue="Season", kde=True, element="step")
plt.title("Rainfall Distribution by Season")
plt.xlabel("Rainfall (mm)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 7. Resource Usage and Irrigation

Resource usage is important for agricultural planning. The project specifically asks us to investigate seasonal differences in resource usage and relationships with outcomes.

In [ ]:
# Cell 20: Irrigation method distribution

irrigation_counts = df_clean["Irrigation_Method"].value_counts()

plt.figure(figsize=(9, 5))
plt.bar(irrigation_counts.index, irrigation_counts.values)
plt.title("Distribution of Irrigation Methods")
plt.xlabel("Irrigation Method")
plt.ylabel("Number of Farms")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 21: Water usage by irrigation method

water_irrigation = (
    df_clean.groupby("Irrigation_Method")["Water_Used_m3"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
sns.barplot(x=water_irrigation.index, y=water_irrigation.values)
plt.title("Average Water Used by Irrigation Method")
plt.xlabel("Irrigation Method")
plt.ylabel("Average Water Used (m³)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

display(water_irrigation.round(2).to_frame("Average Water Used (m³)"))

In [ ]:
# Cell 22: Water efficiency across seasons

water_eff = df_clean.groupby("Season")["Water_Efficiency_t_per_1000m3"].mean()

plt.figure(figsize=(9, 5))
sns.barplot(x=water_eff.index, y=water_eff.values)
plt.title("Average Water Efficiency by Season")
plt.xlabel("Season")
plt.ylabel("Water Efficiency (Tonnes per 1000 m³)")
plt.tight_layout()
plt.show()

## 8. Relationship Analysis

Scatter plots help us investigate whether environmental/resource variables have visible relationships with agricultural yield. Correlation analysis provides a numerical summary of linear relationships.

In [ ]:
# Cell 23: Rainfall vs yield — Seaborn scatter plot

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_clean,
    x="Rainfall_mm",
    y="Yield_Tonnes_Ha",
    hue="Season",
    alpha=0.65
)
plt.title("Rainfall vs Yield")
plt.xlabel("Rainfall (mm)")
plt.ylabel("Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 24: Soil moisture vs yield — Seaborn regression plot

plt.figure(figsize=(10, 6))
sns.regplot(
    data=df_clean,
    x="Soil_Moisture_pct",
    y="Yield_Tonnes_Ha",
    scatter_kws={"alpha": 0.35},
    line_kws={"linewidth": 2}
)
plt.title("Soil Moisture vs Yield")
plt.xlabel("Soil Moisture (%)")
plt.ylabel("Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 25: Temperature vs yield

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_clean,
    x="Avg_Temperature_C",
    y="Yield_Tonnes_Ha",
    hue="Season",
    alpha=0.65
)
plt.title("Average Temperature vs Yield")
plt.xlabel("Average Temperature (°C)")
plt.ylabel("Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 26: Correlation heatmap for numerical variables

corr_cols = [
    "Rainfall_mm", "Avg_Temperature_C", "Humidity_pct",
    "Sunlight_Hours_Day", "Soil_pH", "Soil_Moisture_pct",
    "Nitrogen_kg_ha", "Phosphorus_kg_ha", "Potassium_kg_ha",
    "Fertilizer_kg_ha", "Pesticide_Litre_ha", "Seed_Quality_Score",
    "Yield_Tonnes_Ha", "Production_Tonnes", "Market_Price_INR_Tonne",
    "Total_Cost_INR", "Revenue_INR", "Profit_INR",
    "Water_Used_m3", "Water_Efficiency_t_per_1000m3",
    "Disease_Pest_Risk_pct"
]

corr = df_clean[corr_cols].corr()

plt.figure(figsize=(15, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.2)
plt.title("Correlation Heatmap of Agricultural Variables")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 27: Strongest correlations with yield

yield_corr = (
    df_clean[corr_cols]
    .corr()["Yield_Tonnes_Ha"]
    .drop("Yield_Tonnes_Ha")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

display(yield_corr.round(3).to_frame("Correlation with Yield"))

## 9. Regional Analysis

The project brief also asks whether seasonal patterns are consistent across regions/categories. We therefore compare average yield across states and inspect season-by-state performance.

In [ ]:
# Cell 28: Average yield by state

state_yield = (
    df_clean.groupby("State")["Yield_Tonnes_Ha"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(11, 6))
sns.barplot(x=state_yield.values, y=state_yield.index)
plt.title("Average Yield by State")
plt.xlabel("Average Yield (Tonnes/Ha)")
plt.ylabel("State")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 29: State-season yield heatmap

state_season = pd.pivot_table(
    df_clean,
    values="Yield_Tonnes_Ha",
    index="State",
    columns="Season",
    aggfunc="mean"
)

plt.figure(figsize=(10, 7))
sns.heatmap(state_season, annot=True, fmt=".2f", cmap="YlOrBr")
plt.title("Average Yield by State and Season")
plt.xlabel("Season")
plt.ylabel("State")
plt.tight_layout()
plt.show()

## 10. Economic Performance

Revenue, cost, market price, and profit are important outcome variables. Profit is especially useful because a season may have reasonable production but still perform poorly economically if costs are high or market prices are unfavorable.

In [ ]:
# Cell 30: Average economic indicators by season

economic_summary = df_clean.groupby("Season").agg(
    Avg_Market_Price=("Market_Price_INR_Tonne", "mean"),
    Avg_Cost=("Total_Cost_INR", "mean"),
    Avg_Revenue=("Revenue_INR", "mean"),
    Avg_Profit=("Profit_INR", "mean")
).round(2)

display(economic_summary)

In [ ]:
# Cell 31: Profit distribution by season — violin plot

plt.figure(figsize=(10, 6))
sns.violinplot(data=df_clean, x="Season", y="Profit_INR", inner="quartile")
plt.title("Profit Distribution Across Seasons")
plt.xlabel("Season")
plt.ylabel("Profit (INR)")
plt.ticklabel_format(style="plain", axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 32: Market price vs profit

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_clean,
    x="Market_Price_INR_Tonne",
    y="Profit_INR",
    hue="Season",
    alpha=0.6
)
plt.title("Market Price vs Profit")
plt.xlabel("Market Price (INR/Tonne)")
plt.ylabel("Profit (INR)")
plt.tight_layout()
plt.show()

## 11. Disease and Pest Risk

Disease/pest risk can affect agricultural outcomes. We compare its seasonal level and examine its relationship with yield.

In [ ]:
# Cell 33: Disease/pest risk by season

risk = df_clean.groupby("Season")["Disease_Pest_Risk_pct"].mean().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=risk.index, y=risk.values)
plt.title("Average Disease/Pest Risk by Season")
plt.xlabel("Season")
plt.ylabel("Disease/Pest Risk (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 34: Disease/pest risk vs yield

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_clean,
    x="Disease_Pest_Risk_pct",
    y="Yield_Tonnes_Ha",
    hue="Season",
    alpha=0.6
)
plt.title("Disease/Pest Risk vs Yield")
plt.xlabel("Disease/Pest Risk (%)")
plt.ylabel("Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()

## 12. Outlier Analysis

Extreme observations should not automatically be deleted. They can represent unusual farms, highly productive cases, measurement issues, or special farming conditions. We therefore identify them first and interpret them cautiously.

In [ ]:
# Cell 35: Identify potential yield outliers using IQR

Q1 = df_clean["Yield_Tonnes_Ha"].quantile(0.25)
Q3 = df_clean["Yield_Tonnes_Ha"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

yield_outliers = df_clean[
    (df_clean["Yield_Tonnes_Ha"] < lower_bound) |
    (df_clean["Yield_Tonnes_Ha"] > upper_bound)
]

print("Q1:", round(Q1, 2))
print("Q3:", round(Q3, 2))
print("IQR:", round(IQR, 2))
print("Lower bound:", round(lower_bound, 2))
print("Upper bound:", round(upper_bound, 2))
print("Potential yield outliers:", len(yield_outliers))

display(
    yield_outliers[
        ["Farm_ID", "State", "District", "Crop", "Season",
         "Yield_Tonnes_Ha", "Production_Tonnes", "Profit_INR"]
    ].sort_values("Yield_Tonnes_Ha", ascending=False).head(10)
)

## 13. Key Findings Generated from the Dataset

The next cell automatically produces a concise set of evidence-based findings from the cleaned dataset rather than manually inventing conclusions.

In [ ]:
# Cell 36: Automatically generate key findings

best_yield_season = season_summary["Avg_Yield"].idxmax()
best_profit_season = season_summary["Avg_Profit"].idxmax()
best_water_season = season_summary["Avg_Water_Efficiency"].idxmax()
highest_risk_season = season_summary["Avg_Disease_Risk"].idxmax()

best_crop = crop_yield.idxmax()
best_state = state_yield.idxmax()

print("KEY FINDINGS")
print("-" * 60)
print(f"1. Highest average yield season: {best_yield_season} ({season_summary.loc[best_yield_season, 'Avg_Yield']:.2f} tonnes/ha)")
print(f"2. Highest average profit season: {best_profit_season} (₹{season_summary.loc[best_profit_season, 'Avg_Profit']:,.2f})")
print(f"3. Highest average water-efficiency season: {best_water_season} ({season_summary.loc[best_water_season, 'Avg_Water_Efficiency']:.2f} tonnes/1000 m³)")
print(f"4. Highest average disease/pest risk season: {highest_risk_season} ({season_summary.loc[highest_risk_season, 'Avg_Disease_Risk']:.2f}%)")
print(f"5. Highest average-yield crop: {best_crop} ({crop_yield.loc[best_crop]:.2f} tonnes/ha)")
print(f"6. Highest average-yield state: {best_state} ({state_yield.loc[best_state]:.2f} tonnes/ha)")

strongest_positive = yield_corr.idxmax()
strongest_negative = yield_corr.idxmin()
print(f"7. Strongest positive linear correlation with yield among selected variables: {strongest_positive} ({yield_corr.loc[strongest_positive]:.3f})")
print(f"8. Strongest negative linear correlation with yield among selected variables: {strongest_negative} ({yield_corr.loc[strongest_negative]:.3f})")

## 14. Data-Driven Recommendations

Based on the analysis, recommendations should be tied to the observed patterns:

1. **Plan season-specific cultivation strategies** based on differences in yield, profitability, water efficiency, and environmental conditions.
2. **Prioritize efficient resource use**, especially where water usage is high but water efficiency is relatively low.
3. **Monitor disease and pest risk** more closely during seasons showing higher average risk.
4. **Evaluate crop-season combinations** instead of selecting crops using overall averages alone.
5. **Consider economic outcomes together with yield** because high production does not necessarily guarantee high profit.
6. **Investigate extreme observations** before removing them, because they may reveal unusually successful or problematic farming conditions.
7. **Use regional insights** to support localized agricultural planning rather than applying one strategy to every state or district.

## 15. Conclusion

This analysis explores seasonal agricultural performance using the provided dataset. It covers data cleaning, descriptive statistics, seasonal comparisons, crop and regional analysis, environmental conditions, resource usage, economic performance, disease/pest risk, correlations, and outlier identification.

The visualizations make seasonal differences easier to understand, while the numerical summaries and correlation analysis provide evidence for the observations. The final recommendations are intended to support better evidence-based seasonal agricultural planning.

**Note:** Correlation indicates association, not causation. The findings should therefore be interpreted as patterns in the available dataset rather than proof that one variable directly causes another.

In [ ]:
# Cell 37: Final summary table for the project report

final_summary = season_summary.copy()
final_summary.columns = [
    "Number of Farms",
    "Average Yield (Tonnes/Ha)",
    "Average Production (Tonnes)",
    "Average Revenue (INR)",
    "Average Profit (INR)",
    "Water Efficiency (Tonnes/1000 m³)",
    "Disease/Pest Risk (%)"
]

display(final_summary)